In [1]:
import base

import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm_notebook as tqdm
import torch

from data.gbm import GBM
import tensor_algebra as ta
import iisignature
import problem2
import optimiser_problem2_new_new as optimiser_problem2
from math import factorial
import pickle

C:\Users\User\anaconda3\Lib\site-packages\torch\__init__.py:1275: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\tensor\python_tensor.cpp:436.)
  _C._set_default_tensor_type(t)


In [3]:
sigma_s = 1e-1
b1 = 1e-3
b3 = 1e-2
sigma2 = 1

params = {'b0': 1, 'b1': b1, 'b2': 1, 'b3': b3,
          'sigma0': 1, 'sigma2': sigma2, 'k': 1e-3, 
          'theta1': 10, 'theta2': 1e-2, 'S0': 100, 'Q0': 10, 'X0': 0, 
          'T': 0.10, 'Nt': 1_000}

env = problem2.environment(**params)

In [4]:
order = 3
nsims = 10_000
W = problem2.simulate_BM_with_drift(env, x0=0, mu=0, sigma=1, nsims=nsims)
path_sig_sim = np.zeros((2**(order+1) - 2, nsims))
for nsim in range(nsims):
    path = np.array([env.timesteps, W[:, nsim]]).T
    path_sig = iisignature.sig(path, order)
    path_sig_sim[:, nsim] = path_sig
mean_sig = np.mean(path_sig_sim, axis=1)
mean_sig = np.insert(mean_sig, 0, 1)
mean_sig = ta.sig_to_tensor(mean_sig, dim = 2, order = order)

In [8]:
time_augmented_path = np.array([env.timesteps, W[:, 0]]).T
iisignature.sig(time_augmented_path, order)

array([1.00000000e-01, 3.39076819e-01, 5.00000000e-03, 1.45726721e-02,
       1.93350099e-02, 5.74865447e-02, 1.66666667e-04, 3.68442652e-04,
       7.20381902e-04, 2.07006144e-03, 6.06559542e-04, 8.01132404e-04,
       2.87746062e-03, 6.49745158e-03])

In [36]:
dim = 2
#path_sig_sim_0 = ta.one(dim = dim)
path_sig_sim_0 = np.zeros((2**(order+1) - 2,))
for i in range(env.Nt):
    #path_increment = np.insert(path_test[i+1] - path_test[i], 0, 1)
    path_increment = path_test[i+1] - path_test[i]
    #path_increment = ta.sig_to_tensor(path_increment, dim = dim, order = 1)
    path_sig_sim_0 = iisignature.sigjoin(path_sig_sim_0, path_increment, order)

In [47]:
time_augmented_path

array([[0.00000000e+00, 0.00000000e+00],
       [1.00000000e-04, 4.64699733e-03],
       [2.00000000e-04, 1.30932586e-02],
       ...,
       [9.98000000e-02, 3.43925317e-01],
       [9.99000000e-02, 3.36385231e-01],
       [1.00000000e-01, 3.39076819e-01]])

In [22]:
len(env.timesteps)

1001

In [40]:
testa = np.array([[1.,2.], [3.,4.]])

In [45]:
testa[1:2,0] - testa[0:1,0]

array([2.])